**Along with finding the best configuration of hyper parameters ,Hyper Parameter Tuning also selects the best model to optimize the prediction**

In [20]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import optuna
import warnings
warnings.filterwarnings('ignore')
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold,cross_val_score

In [4]:
from sklearn.datasets import load_breast_cancer

In [5]:
data = load_breast_cancer()

In [6]:
X = data.data
X

array([[1.799e+01, 1.038e+01, 1.228e+02, ..., 2.654e-01, 4.601e-01,
        1.189e-01],
       [2.057e+01, 1.777e+01, 1.329e+02, ..., 1.860e-01, 2.750e-01,
        8.902e-02],
       [1.969e+01, 2.125e+01, 1.300e+02, ..., 2.430e-01, 3.613e-01,
        8.758e-02],
       ...,
       [1.660e+01, 2.808e+01, 1.083e+02, ..., 1.418e-01, 2.218e-01,
        7.820e-02],
       [2.060e+01, 2.933e+01, 1.401e+02, ..., 2.650e-01, 4.087e-01,
        1.240e-01],
       [7.760e+00, 2.454e+01, 4.792e+01, ..., 0.000e+00, 2.871e-01,
        7.039e-02]])

In [7]:
y = data.target
y

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0,
       1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0,
       1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0,
       0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1,
       1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0,
       0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0,
       1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1,
       1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0,

In [8]:
df =pd.DataFrame(X,columns=data.feature_names)
df['target'] = y
df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [11]:
#Building a model using DecisionTreeClassifier to solve the binary classification problem
#Feature and target
X.shape,y.shape

((569, 30), (569,))

In [12]:
#splitting the data into train and test
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,stratify=y,test_size=0.2,random_state=42)
X_train.shape,X_test.shape,y_train.shape,y_test.shape

((455, 30), (114, 30), (455,), (114,))

In [13]:
#model building
dt = DecisionTreeClassifier()
dt

DecisionTreeClassifier()

In [14]:
dt.fit(X_train,y_train)

DecisionTreeClassifier()

In [15]:
dt.score(X_train,y_train)

1.0

In [16]:
y_pred = dt.predict(X_test)
y_pred

array([0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1,
       1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0,
       0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1,
       0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0,
       1, 0, 1, 1])

In [17]:
from sklearn.metrics import accuracy_score

In [18]:
accuracy_score(y_test,y_pred)

0.9210526315789473

### HPT

In [26]:
#Objective
def objective(trail):
    #hyperparameter
    max_depth=trail.suggest_int('max_depth',2,30)
    min_sam_split = trail.suggest_int('min_samples_split',2,50)
    min_sam_leaf = trail.suggest_int('min_samples_leaf',1,20)
    criterion = trail.suggest_categorical('criterion',['gini', 'entropy', 'log_loss'])
    #model
    dt = DecisionTreeClassifier(criterion=criterion,max_depth=max_depth,min_samples_leaf=min_sam_leaf,min_samples_split=min_sam_split)
    #metrics
    skf = StratifiedKFold(n_splits=5,shuffle=True)
    score = cross_val_score(dt,X_train,y_train,scoring='accuracy',cv=skf).mean()
    return score

In [27]:
#Create a study
study = optuna.create_study(study_name='DT_study',direction='maximize',sampler=optuna.samplers.RandomSampler())
study

[I 2026-01-30 15:58:57,436] A new study created in memory with name: DT_study


In [28]:
#Optimize the study
study.optimize(objective,n_trials=40)

[I 2026-01-30 15:58:58,301] Trial 0 finished with value: 0.9010989010989011 and parameters: {'max_depth': 2, 'min_samples_split': 35, 'min_samples_leaf': 15, 'criterion': 'entropy'}. Best is trial 0 with value: 0.9010989010989011.
[I 2026-01-30 15:58:58,521] Trial 1 finished with value: 0.9252747252747252 and parameters: {'max_depth': 26, 'min_samples_split': 14, 'min_samples_leaf': 6, 'criterion': 'gini'}. Best is trial 1 with value: 0.9252747252747252.
[I 2026-01-30 15:58:58,620] Trial 2 finished with value: 0.9120879120879121 and parameters: {'max_depth': 27, 'min_samples_split': 2, 'min_samples_leaf': 13, 'criterion': 'gini'}. Best is trial 1 with value: 0.9252747252747252.
[I 2026-01-30 15:58:58,700] Trial 3 finished with value: 0.9428571428571428 and parameters: {'max_depth': 30, 'min_samples_split': 13, 'min_samples_leaf': 6, 'criterion': 'gini'}. Best is trial 3 with value: 0.9428571428571428.
[I 2026-01-30 15:58:58,806] Trial 4 finished with value: 0.8945054945054945 and param

In [29]:
study.best_params

{'max_depth': 30,
 'min_samples_split': 13,
 'min_samples_leaf': 6,
 'criterion': 'gini'}

In [32]:
study.best_value

0.9428571428571428

- Creating a study using Grid_sampler()

In [42]:
#For grid sampler we have create a search space
search_space = {
    'criterion':['gini','entropy','log_loss'],
    'max_depth':[2,5,10,15,20,25],
    'min_samples_split': [2,5,10,15,20],
    'min_samples_leaf':[2,5,10,15,20,25]
}

In [43]:
study = optuna.create_study(study_name='Dt_study',direction='maximize',sampler=optuna.samplers.GridSampler(search_space))

[I 2026-01-30 16:09:36,066] A new study created in memory with name: Dt_study


In [44]:
study.optimize(objective,n_trials=40)

[I 2026-01-30 16:09:37,554] Trial 0 finished with value: 0.9274725274725274 and parameters: {'max_depth': 10, 'min_samples_split': 5, 'min_samples_leaf': 10, 'criterion': 'gini'}. Best is trial 0 with value: 0.9274725274725274.
[I 2026-01-30 16:09:37,661] Trial 1 finished with value: 0.9384615384615385 and parameters: {'max_depth': 5, 'min_samples_split': 5, 'min_samples_leaf': 2, 'criterion': 'log_loss'}. Best is trial 1 with value: 0.9384615384615385.
[I 2026-01-30 16:09:37,751] Trial 2 finished with value: 0.9340659340659341 and parameters: {'max_depth': 20, 'min_samples_split': 15, 'min_samples_leaf': 10, 'criterion': 'log_loss'}. Best is trial 1 with value: 0.9384615384615385.
[I 2026-01-30 16:09:37,846] Trial 3 finished with value: 0.923076923076923 and parameters: {'max_depth': 5, 'min_samples_split': 15, 'min_samples_leaf': 20, 'criterion': 'entropy'}. Best is trial 1 with value: 0.9384615384615385.
[I 2026-01-30 16:09:37,917] Trial 4 finished with value: 0.9186813186813186 and

In [45]:
study.best_params

{'max_depth': 20,
 'min_samples_split': 10,
 'min_samples_leaf': 10,
 'criterion': 'log_loss'}

In [46]:
study.best_value

0.9406593406593405

- Creating a study with sequential search TPE(Default)

In [47]:
study = optuna.create_study(study_name='Dt_study',direction='maximize')

[I 2026-01-30 16:13:04,562] A new study created in memory with name: Dt_study


In [48]:
study.optimize(objective,n_trials=40)

[I 2026-01-30 16:13:20,008] Trial 0 finished with value: 0.9252747252747252 and parameters: {'max_depth': 18, 'min_samples_split': 2, 'min_samples_leaf': 18, 'criterion': 'gini'}. Best is trial 0 with value: 0.9252747252747252.
[I 2026-01-30 16:13:20,198] Trial 1 finished with value: 0.9208791208791208 and parameters: {'max_depth': 4, 'min_samples_split': 12, 'min_samples_leaf': 15, 'criterion': 'log_loss'}. Best is trial 0 with value: 0.9252747252747252.
[I 2026-01-30 16:13:20,354] Trial 2 finished with value: 0.9208791208791209 and parameters: {'max_depth': 5, 'min_samples_split': 15, 'min_samples_leaf': 17, 'criterion': 'log_loss'}. Best is trial 0 with value: 0.9252747252747252.
[I 2026-01-30 16:13:20,522] Trial 3 finished with value: 0.9230769230769231 and parameters: {'max_depth': 27, 'min_samples_split': 36, 'min_samples_leaf': 14, 'criterion': 'entropy'}. Best is trial 0 with value: 0.9252747252747252.
[I 2026-01-30 16:13:20,675] Trial 4 finished with value: 0.9142857142857144 

In [49]:
study.best_params

{'max_depth': 11,
 'min_samples_split': 14,
 'min_samples_leaf': 7,
 'criterion': 'log_loss'}

In [50]:
study.best_value

0.9406593406593406

In [57]:
dt = DecisionTreeClassifier(**study.best_params)
dt

DecisionTreeClassifier(criterion='log_loss', max_depth=11, min_samples_leaf=7,
                       min_samples_split=14)

In [58]:
dt.fit(X_train,y_train)

DecisionTreeClassifier(criterion='log_loss', max_depth=11, min_samples_leaf=7,
                       min_samples_split=14)

In [59]:
dt.score(X_train,y_train)

0.9648351648351648

In [60]:
y_pred = dt.predict(X_test)
y_pred

array([0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0,
       1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0,
       0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1,
       1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0,
       1, 0, 1, 1])

In [61]:
accuracy_score(y_pred,y_test)

0.956140350877193

- For comparing the best model also using knn to build the model as my second model

In [51]:
from sklearn.neighbors import KNeighborsClassifier

In [52]:
knn = KNeighborsClassifier(n_neighbors=5,weights='distance')
knn

KNeighborsClassifier(weights='distance')

In [53]:
knn.fit(X_train,y_train)

KNeighborsClassifier(weights='distance')

In [54]:
knn.score(X_train,y_train)

1.0

In [55]:
y_pred = knn.predict(X_test)
y_pred

array([0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0,
       1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0,
       0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1,
       1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0,
       1, 0, 1, 1])

In [56]:
accuracy_score(y_pred,y_test)

0.9122807017543859

In [62]:
# Objective
def objective(trial):
  # Levels of Hyperparameters
  model = trial.suggest_categorical('model', ['knn', 'dt'])
  if model == 'knn':
    n_neighbors = trial.suggest_int('n_neighbors', 3, 30)
    weights = trial.suggest_categorical('weights', ['uniform', 'distance'])
    clf = KNeighborsClassifier(n_neighbors=n_neighbors, weights = weights)
  else:
    criterion = trial.suggest_categorical('criterion', ['gini', 'entropy', 'log_loss'])
    max_depth = trial.suggest_int('max_depth', 2, 30)
    min_sam_split = trial.suggest_int('min_samples_split', 2, 50)
    min_sam_leaf = trial.suggest_int('min_samples_leaf', 1, 20)
    clf = DecisionTreeClassifier(criterion= criterion, max_depth= max_depth, min_samples_split = min_sam_split, min_samples_leaf = min_sam_leaf)
  # Metric
  skf = StratifiedKFold(n_splits = 5, shuffle = True)
  score = cross_val_score(clf, X_train, y_train, scoring = 'accuracy', cv = skf).mean()
  return score

In [65]:
study = optuna.create_study(direction='maximize',study_name='DT_study')
study

[I 2026-01-30 16:46:39,764] A new study created in memory with name: DT_study


In [67]:
study.optimize(objective,n_trials=50)

[I 2026-01-30 16:47:38,958] Trial 0 finished with value: 0.9230769230769231 and parameters: {'model': 'knn', 'n_neighbors': 25, 'weights': 'uniform'}. Best is trial 0 with value: 0.9230769230769231.
[I 2026-01-30 16:47:39,029] Trial 1 finished with value: 0.9142857142857143 and parameters: {'model': 'dt', 'criterion': 'log_loss', 'max_depth': 25, 'min_samples_split': 49, 'min_samples_leaf': 8}. Best is trial 0 with value: 0.9230769230769231.
[I 2026-01-30 16:47:39,096] Trial 2 finished with value: 0.9318681318681319 and parameters: {'model': 'dt', 'criterion': 'gini', 'max_depth': 18, 'min_samples_split': 25, 'min_samples_leaf': 15}. Best is trial 2 with value: 0.9318681318681319.
[I 2026-01-30 16:47:39,160] Trial 3 finished with value: 0.9406593406593406 and parameters: {'model': 'knn', 'n_neighbors': 6, 'weights': 'distance'}. Best is trial 3 with value: 0.9406593406593406.
[I 2026-01-30 16:47:39,207] Trial 4 finished with value: 0.9252747252747252 and parameters: {'model': 'knn', 'n

In [68]:
study.best_params

{'model': 'knn', 'n_neighbors': 5, 'weights': 'distance'}

- Between DecisionTreeClassifier and KNeighborsClassifier ,Knn is the best model from HPT 

In [69]:
study.best_value

0.9428571428571428

In [71]:
knn = KNeighborsClassifier(n_neighbors=5,weights='distance')
knn

KNeighborsClassifier(weights='distance')

In [72]:
knn.fit(X_train,y_train)

KNeighborsClassifier(weights='distance')

In [73]:
knn.score(X_train,y_train)

1.0

In [74]:
y_pred = knn.predict(X_test)
y_pred

array([0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0,
       1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0,
       0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1,
       1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0,
       1, 0, 1, 1])

In [75]:
accuracy_score(y_test,y_pred)

0.9122807017543859